## Data Transformation

In [27]:
import os
from openai import OpenAI
from groq import Groq
from litellm import completion
from dotenv import load_dotenv
import json
from pricer.batch import Batch
from pricer.items import Item

load_dotenv(override=True)

openrouter = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

In [68]:
username = "TumeloKonaite"
dataset = f"{username}/items_raw_lite"

train, val, test = Item.from_hub(dataset)

items = train + val + test

print(f"Loaded {len(items):,} items")
print(items[0])

Loaded 22,000 items
title='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)' category='Tools_and_Home_Improvement' price=64.3 full='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)\n[\'From the Manufacturer\', "When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we\'re the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid"]\n[\'Interior half only\', \'Requires F58 to complete handle set\', \'Non handed knob style\', \'4" minimum center to center door prep required for this two piece model.\', \'Lifetime Mechanical and Finish Warranty\']\n{"Material": "Metal", "Brand": "", "Color": "Oil Rubbed Bronze", "Exterior Finish": "Bronze", "Special Feature": "Easy to Install", "Age Range (Description)": "Adult", "Included Components": "Deadbolt, Knob", "Item Wei

In [69]:
# Give every item an id

for index, item in enumerate(items):
    item.id = index

In [70]:


SYSTEM_PROMPT = """Create a concise description of a product. Respond only in this format. Do not include part numbers.
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features"""

In [31]:
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
response = completion(messages=messages, model="groq/openai/gpt-oss-20b", reasoning_effort="low")

print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")


Title: Schlage F59 & 613 Interior Half Knob with Deadbolt – Oil Rubbed Bronze  
Category: Hardware  
Brand: Schlage  
Description: A high‑security interior half knob set that includes a deadbolt, designed for seamless installation on 4″‑center doors.  
Details: Made from durable oil‑rubbed bronze metal, it features a solid‑metal construction, easy installation, and a lifetime mechanical and finish warranty.

Input tokens: 446
Output tokens: 110
Cost: 0.007 cents


In [101]:
MODEL = "gpt-4.1 nano"  #"gpt-4o-mini"  # OpenAI Batch API-supported model

In [102]:
def make_jsonl(item):
    body = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": item.full},
        ],
    }
    line = {"custom_id": str(item.id), "method": "POST", "url": "/v1/chat/completions", "body": body}
    return json.dumps(line)

In [103]:
items[0]

<Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only) = $64.3>

In [104]:
make_jsonl(items[0])

'{"custom_id": "0", "method": "POST", "url": "/v1/chat/completions", "body": {"model": "gpt-4.1 nano", "messages": [{"role": "system", "content": "Create a concise description of a product. Respond only in this format. Do not include part numbers.\\nTitle: Rewritten short precise title\\nCategory: eg Electronics\\nBrand: Brand name\\nDescription: 1 sentence description\\nDetails: 1 sentence on features"}, {"role": "user", "content": "Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)\\n[\'From the Manufacturer\', \\"When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we\'re the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid\\"]\\n[\'Interior half only\', \'Requires F58 to complete handle set\', \'Non handed knob style\', \'4\\" minimum center to center door prep required for this two piece model.\

In [105]:

def make_file(start, end, filename):
    batch_file = filename
    folder = os.path.dirname(batch_file)
    if folder:
        os.makedirs(folder, exist_ok=True)
    with open(batch_file, "w", encoding="utf-8") as f:
        for i in range(start, end):
            f.write(make_jsonl(items[i]))
            f.write("\n")

In [106]:
make_file(0, 200, "jsonl/0_200.jsonl")

In [107]:
import os
openai = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY")
)

In [108]:
with open("jsonl/0_200.jsonl", "rb") as f:
    response = openai.files.create(file=f, purpose="batch")

print("Uploaded file ID:", response.id)
response

Uploaded file ID: file-66tm7H6PkX4fFKcYyLMDUL


FileObject(id='file-66tm7H6PkX4fFKcYyLMDUL', bytes=446019, created_at=1772569860, filename='0_200.jsonl', object='file', purpose='batch', status='processed', expires_at=1775161860, status_details=None)

In [109]:
file_id = response.id
file_id

'file-66tm7H6PkX4fFKcYyLMDUL'

In [110]:
response = openai.batches.create(completion_window="24h", endpoint="/v1/chat/completions", input_file_id=file_id)
response

Batch(id='batch_69a745b14c608190892f99d60ec5e5cd', completion_window='24h', created_at=1772570033, endpoint='/v1/chat/completions', input_file_id='file-66tm7H6PkX4fFKcYyLMDUL', object='batch', status='validating', cancelled_at=None, cancelling_at=None, completed_at=None, error_file_id=None, errors=None, expired_at=None, expires_at=1772656433, failed_at=None, finalizing_at=None, in_progress_at=None, metadata=None, model=None, output_file_id=None, request_counts=BatchRequestCounts(completed=0, failed=0, total=0), usage=BatchUsage(input_tokens=0, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=0, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=0))

In [111]:
result = openai.batches.retrieve(response.id)
result

Batch(id='batch_69a745b14c608190892f99d60ec5e5cd', completion_window='24h', created_at=1772570033, endpoint='/v1/chat/completions', input_file_id='file-66tm7H6PkX4fFKcYyLMDUL', object='batch', status='failed', cancelled_at=None, cancelling_at=None, completed_at=None, error_file_id=None, errors=Errors(data=[BatchError(code='model_not_found', line=1, message="The provided model 'gpt-4.1 nano' is not supported by the Batch API.", param='body.model'), BatchError(code='model_not_found', line=2, message="The provided model 'gpt-4.1 nano' is not supported by the Batch API.", param='body.model'), BatchError(code='model_not_found', line=3, message="The provided model 'gpt-4.1 nano' is not supported by the Batch API.", param='body.model'), BatchError(code='model_not_found', line=4, message="The provided model 'gpt-4.1 nano' is not supported by the Batch API.", param='body.model'), BatchError(code='model_not_found', line=5, message="The provided model 'gpt-4.1 nano' is not supported by the Batch 

In [ ]:
import time

batch_id = response.id
max_wait_seconds = 1800
poll_every_seconds = 15
elapsed = 0

while True:
    result = openai.batches.retrieve(batch_id)
    counts = result.request_counts
    print(f"status={result.status} completed={counts.completed}/{counts.total} failed={counts.failed}")

    if result.status == "completed":
        out = openai.files.content(result.output_file_id)
        out.write_to_file("jsonl/batch_results.jsonl")
        print("Saved: jsonl/batch_results.jsonl")
        break

    if result.status in {"failed", "expired", "cancelled"}:
        if result.error_file_id:
            err = openai.files.content(result.error_file_id)
            err.write_to_file("jsonl/batch_errors.jsonl")
            print("Saved: jsonl/batch_errors.jsonl")
        elif result.errors and getattr(result.errors, "data", None):
            for e in result.errors.data[:5]:
                print(f"line={e.line} code={e.code} message={e.message}")
        raise RuntimeError(f"Batch ended with status={result.status}")

    if elapsed >= max_wait_seconds:
        raise TimeoutError(f"Batch still {result.status} after {max_wait_seconds}s")

    time.sleep(poll_every_seconds)
    elapsed += poll_every_seconds

ValueError: Expected a non-empty value for `file_id` but received None